# v40 — быстрые G1/G2 и BN

Run All на текущем Mac. Без optimizer updates, без изменения MVP, bbox и порога.
17 вариантов fusion на готовых признаках: выбор на calibration, затем только контроль и победитель на validation.
BN: R1 и N1, до/после одного прохода **только primary train**. Полный набор экспериментов GPU — в v41.
Время не ограничено; быстрее обучения, но BN требует чтения изображений и encoder-forward.


In [1]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['ORT_DISABLE_TELEMETRY'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['XFORMERS_DISABLED'] = '1'
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'training/research_gpu.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook внутри клона Car-classification-MSK / fine-tuning')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
INPUTS = ROOT / 'research_transfer/v41_inputs'
print('Python:', sys.executable)
print('Repo:', ROOT)
print('Inputs:', INPUTS)
if not (INPUTS / 'inputs.json').is_file():
    raise FileNotFoundError('Распакуй v41_inputs.zip в research_transfer/v41_inputs. См. README варианта.')
RUN_NAME = 'quick_v1'
DEVICE = 'mps'  # 'cpu' только если MPS недоступен; автоматической подмены нет.


Python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Repo: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK
Inputs: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/research_transfer/v41_inputs


In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_parallel_research.py'], cwd=ROOT, env=os.environ.copy(), check=True)

...........................                                              [100%]
27 passed in 5.59s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_parallel_research.py'], returncode=0)

In [3]:
from training.research_quick import run
summary = run(INPUTS, RUN_NAME, DEVICE)


calibration V25_control: mAP=0.826410
calibration N1_G1_w02: mAP=0.826436
calibration N1_G1_w05: mAP=0.826246
calibration N1_G1_w10: mAP=0.826718
calibration N1_G1_w20: mAP=0.826877
calibration N1_G2_w02: mAP=0.826003
calibration N1_G2_w05: mAP=0.826003
calibration N1_G2_w10: mAP=0.825947
calibration N1_G2_w20: mAP=0.825809
calibration T12_G1_w02: mAP=0.826477
calibration T12_G1_w05: mAP=0.827046
calibration T12_G1_w10: mAP=0.828935
calibration T12_G1_w20: mAP=0.830076
calibration T12_G2_w02: mAP=0.826394
calibration T12_G2_w05: mAP=0.826655
calibration T12_G2_w10: mAP=0.826636
calibration T12_G2_w20: mAP=0.827403
validation V25_control: mAP=0.828957
validation T12_G1_w20: mAP=0.824723
PREFLIGHT: 740 train-ID, 185 holdout-ID | mps  | frozen primary only
CONTROL: B0
FEATURES 16/1036
FEATURES 272/1036
FEATURES 528/1036
FEATURES 784/1036
FEATURES 1036/1036
CONTROL: R1_20260915
FEATURES 16/1036
FEATURES 272/1036
FEATURES 528/1036
FEATURES 784/1036
FEATURES 1036/1036
CONTROL: R1_20260916
FE

In [4]:
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Отдать на анализ:', ROOT / f'OSNet-AIN-x1.0/variant_40_quick_diagnostics/{RUN_NAME}_analysis.zip')


{
  "status": "complete",
  "elapsed_seconds": 307.0149459160166,
  "fusion_validation": {
    "V25_control": {
      "n_scored": 247,
      "n_openset_excluded": 62,
      "mAP@10": 0.8289573904235558,
      "Rank-1": 0.8299595141700404,
      "Rank-5": 0.8785425101214575
    },
    "T12_G1_w20": {
      "n_scored": 247,
      "n_openset_excluded": 62,
      "mAP@10": 0.8247228113017586,
      "Rank-1": 0.8259109311740891,
      "Rank-5": 0.8744939271255061
    }
  },
  "bn_primary": {
    "R1_20260915_unchanged": {
      "raw": 0.8328422682886969,
      "graph": 0.8440989714203999,
      "control": 0.8494544927580643,
      "pre_graph_10": 0.8479975000510714,
      "G1": 0.8499229969765684,
      "G2": 0.8495929645632027
    },
    "R1_20260915_recalibrated": {
      "raw": 0.8267371538204871,
      "graph": 0.8410109280198567,
      "control": 0.8494544927580643,
      "pre_graph_10": 0.8469259225211606,
      "G1": 0.8499048836251218,
      "G2": 0.8495339770042151
    },
    "N1_u